In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from bs4 import BeautifulSoup
import re

sns.set_style('darkgrid')


In [6]:
df = pd.read_csv('IMDB Dataset.csv')

In [7]:
df.sample(5)

,review,sentiment
1140,"In my Lit. class we've just finished the book,...",negative
25095,"""Who Loves The Sun"" works its way through some...",positive
25912,"""New Best Friend"" is another entry in the ""ste...",negative
18175,CIA analyst Douglas Freeman (Gyllenhaal) gets ...,positive
5078,"After a humiliating experience on an airplane,...",negative


In [9]:
df_pos = df[df['sentiment']=='positive'][:5000]
df_neg = df[df['sentiment']=='negative'][:5000]

df_reviews = pd.concat([df_pos, df_neg ])

In [10]:
from sklearn.model_selection import train_test_split


In [11]:
train,test = train_test_split(df_reviews,test_size =0.33,random_state=42)

In [12]:
train_x, train_y = train['review'], train['sentiment']
test_x, test_y = test['review'], test['sentiment']

In [13]:
train_y.value_counts()

,count
sentiment,
negative,3378
positive,3322


In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [15]:
tfidf = TfidfVectorizer(stop_words='english')
train_x_vector = tfidf.fit_transform(train_x)
test_x_vector = tfidf.transform(test_x)

In [16]:
train_x.shape

(6700,)

In [19]:
train_x_vector.shape

(6700, 44107)

In [18]:
type(train_x_vector)

scipy.sparse._csr.csr_matrix

In [20]:
primera_resenia = pd.DataFrame.sparse.from_spmatrix(train_x_vector,
                                  index=train_x.index,
                                  columns=tfidf.get_feature_names_out()).iloc[0]

In [21]:
primera_resenia

,6746
00,0
000,0
007,0
00am,0
00s,0
...,...
ísnt,0
île,0
önsjön,0
über,0


In [22]:
train_x.iloc[0]

"I happened to rent this movie with my sister in hopes of watching a great entertaining movie, that was humorous, however my expectations were let down. This movie was beyond disgusting and revolting for a PG-13 movie, this should have been rated R for the many mature references that went on in this movie. I wouldn't recommend allowing a 13 year old teen see this.<br /><br />Even if no one under the age of 17 is watching this movie, beware of a truly stupid movie, there's no humor in the movie, just a bunch of disgusting sexual references including a small touch of pedophilia, something that shouldn't even be joked about. <br /><br />I would like to know what happened to PG-13 movies, that were actually safe for actual a 13 year old? This is beyond a deplorable movie and should be re-rated."

In [23]:
primera_resenia[primera_resenia != 0]

,6746
13,0.45849
17,0.12824
actual,0.091601
actually,0.061461
age,0.088765
allowing,0.12824
beware,0.143046
br,0.124945
bunch,0.093128
deplorable,0.168137


In [24]:
train_x.iloc[0]

"I happened to rent this movie with my sister in hopes of watching a great entertaining movie, that was humorous, however my expectations were let down. This movie was beyond disgusting and revolting for a PG-13 movie, this should have been rated R for the many mature references that went on in this movie. I wouldn't recommend allowing a 13 year old teen see this.<br /><br />Even if no one under the age of 17 is watching this movie, beware of a truly stupid movie, there's no humor in the movie, just a bunch of disgusting sexual references including a small touch of pedophilia, something that shouldn't even be joked about. <br /><br />I would like to know what happened to PG-13 movies, that were actually safe for actual a 13 year old? This is beyond a deplorable movie and should be re-rated."

In [25]:
from sklearn.svm import SVC
svc = SVC(kernel='linear')
svc.fit(train_x_vector, train_y)

SVC(kernel='linear')

In [26]:
print(svc.predict(tfidf.transform(['A good movie'])))
print(svc.predict(tfidf.transform(['An excellent movie'])))
print(svc.predict(tfidf.transform(['I did not like this movie at all I gave this movie away'])))

['positive']
['positive']
['negative']


In [28]:
print(svc.score(test_x_vector, test_y))

0.8706060606060606


In [30]:
from sklearn.metrics import f1_score

f1_score(test_y,svc.predict(test_x_vector),
          labels = ['positive','negative'],average=None)

array([0.87400413, 0.86701962])

In [31]:
from sklearn.metrics import classification_report

print(classification_report(test_y,
                            svc.predict(test_x_vector),
                            labels = ['positive','negative']))

              precision    recall  f1-score   support

    positive       0.87      0.88      0.87      1678
    negative       0.88      0.86      0.87      1622

    accuracy                           0.87      3300
   macro avg       0.87      0.87      0.87      3300
weighted avg       0.87      0.87      0.87      3300



In [32]:
from sklearn.metrics import confusion_matrix

conf_mat = confusion_matrix(test_y,
                           svc.predict(test_x_vector),
                           labels = ['positive', 'negative'])
conf_mat

array([[1481,  197],
       [ 230, 1392]])

In [33]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(train_x_vector, train_y)

log_pred = log_reg.predict(test_x_vector)

print("Accuracy Logistic Regression:", accuracy_score(test_y, log_pred))
print("F1 Score positivo:", f1_score(test_y, log_pred, pos_label='positive'))
print("F1 Score negativo:", f1_score(test_y, log_pred, pos_label='negative'))

print(classification_report(test_y, log_pred))


Accuracy Logistic Regression: 0.8718181818181818
F1 Score positivo: 0.8748890860692103
F1 Score negativo: 0.8685927306616962
              precision    recall  f1-score   support

    negative       0.88      0.86      0.87      1622
    positive       0.87      0.88      0.87      1678

    accuracy                           0.87      3300
   macro avg       0.87      0.87      0.87      3300
weighted avg       0.87      0.87      0.87      3300



In [35]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

nb = MultinomialNB()
nb.fit(train_x_vector, train_y)

nb_pred = nb.predict(test_x_vector)

print("Accuracy Naive Bayes:", accuracy_score(test_y, nb_pred))
print("F1 Score positivo:", f1_score(test_y, nb_pred, pos_label='positive'))
print("F1 Score negativo:", f1_score(test_y, nb_pred, pos_label='negative'))

print(classification_report(test_y, nb_pred))


Accuracy Naive Bayes: 0.8542424242424242
F1 Score positivo: 0.8475435816164818
F1 Score negativo: 0.8603773584905661
              precision    recall  f1-score   support

    negative       0.81      0.91      0.86      1622
    positive       0.91      0.80      0.85      1678

    accuracy                           0.85      3300
   macro avg       0.86      0.86      0.85      3300
weighted avg       0.86      0.85      0.85      3300



In [36]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

svm_pred = svc.predict(test_x_vector)
log_pred = log_reg.predict(test_x_vector)
nb_pred = nb.predict(test_x_vector)

resultados = pd.DataFrame({
    'Modelo': ['SVM', 'Logistic Regression', 'Naive Bayes'],
    'Accuracy': [
        accuracy_score(test_y, svm_pred),
        accuracy_score(test_y, log_pred),
        accuracy_score(test_y, nb_pred)
    ],
    'F1 positivo': [
        f1_score(test_y, svm_pred, pos_label='positive'),
        f1_score(test_y, log_pred, pos_label='positive'),
        f1_score(test_y, nb_pred, pos_label='positive')
    ],
    'F1 negativo': [
        f1_score(test_y, svm_pred, pos_label='negative'),
        f1_score(test_y, log_pred, pos_label='negative'),
        f1_score(test_y, nb_pred, pos_label='negative')
    ]
})

resultados


,Modelo,Accuracy,F1 positivo,F1 negativo
0,SVM,0.870606,0.874004,0.867020
1,Logistic Regression,0.871818,0.874889,0.868593
2,Naive Bayes,0.854242,0.847544,0.860377


In [37]:
frases = [
    "This movie was amazing, I loved it",
    "The film was boring and too long",
    "The acting was excellent but the story was weak",
    "I would not recommend this movie",
    "It was one of the best movies I have seen"
]

for frase in frases:
    prediccion = log_reg.predict(tfidf.transform([frase]))
    print(frase, "->", prediccion[0])


This movie was amazing, I loved it -> positive
The film was boring and too long -> negative
The acting was excellent but the story was weak -> positive
I would not recommend this movie -> positive
It was one of the best movies I have seen -> positive


## Conclusión

En este proyecto se realizó un análisis de sentimientos usando reseñas de películas del dataset IMDB.  
Primero se limpiaron y vectorizaron los textos usando TF-IDF. Después se entrenaron distintos modelos de clasificación, incluyendo SVM, Regresión Logística y Naive Bayes.

Los modelos obtuvieron un desempeño cercano al 87% de accuracy. SVM y Regresión Logística mostraron resultados muy similares, con buenos valores de precisión, recall y F1-score tanto para reseñas positivas como negativas.

La Regresión Logística obtuvo un accuracy de aproximadamente 0.8718, por lo que puede considerarse uno de los mejores modelos en esta prueba. Además, es un modelo rápido, interpretable y adecuado para clasificación de texto.

En general, los resultados muestran que es posible clasificar reseñas de películas como positivas o negativas con buen rendimiento utilizando técnicas clásicas de machine learning y representación TF-IDF.
